#### mlflow setup and establishing connection to dagshub

In [1]:
import mlflow
import dagshub

mlflow.set_experiment("House prices")
mlflow.set_tracking_uri("https://dagshub.com/gbera23-dev/Machine-Learning.mlflow")

dagshub.init(repo_owner='gbera23-dev', repo_name='Machine-Learning', mlflow=True)

/home/giga/FREEUNI/semester_6/ML/Machine-Learning/Assignment1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/04/10 02:06:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/10 02:06:08 INFO mlflow.store.db.utils: Updating database tables
2026/04/10 02:06:09 INFO mlflow.tracking.fluent: Experiment with name 'House prices' does not exist. Creating a new experiment.


Accessing as gbera23-dev

Initialized MLflow to track repo "gbera23-dev/Machine-Learning"

Repository gbera23-dev/Machine-Learning initialized!

#### Importing necessary libraries and loading data into a notebook

In [2]:
import pandas as pd
import numpy as np

train_data_file_path = "house-prices-advanced-regression-techniques/train.csv"
dataset_df = pd.read_csv(train_data_file_path).copy(True)

FileNotFoundError: [Errno 2] No such file or directory: 'house-prices-advanced-regression-techniques/train.csv'

#### Starting up new preprocessing run

In [ ]:
mlflow.start_run(run_name="decision_tree_optimized_model_1")

#### Exploratory Data Analysis

##### Running a basic check

In [ ]:
dataset_df.describe()

##### Checking out Categorical and Numerical columns

In [ ]:
numerical_part_df = dataset_df.select_dtypes(include=['int64', 'float64'])
categorical_part_df = dataset_df.select_dtypes(exclude=['int64', 'float64'])

num_numerical = numerical_part_df.columns.size
num_categorical = categorical_part_df.columns.size

print(f"number of numerical columns: {num_numerical}")
print(f"number of categorical columns: {num_categorical}")

mlflow.log_param("num_numerical", num_numerical)
mlflow.log_param("num_categorical", num_categorical)

##### Analyzing numerical columns

In [ ]:
numerical_part_df.describe()

##### Analyzing categorical columns

In [ ]:
cardinality_dict = {}
for col in categorical_part_df.columns:
    cardinality_dict[col] = categorical_part_df[col].nunique()

cardinality_dict = dict(sorted(cardinality_dict.items(), key=lambda item: item[1], reverse=True))
print(cardinality_dict)

top_5 = dict(list(cardinality_dict.items())[:5])
mlflow.log_param("max cardinality", max(cardinality_dict.values()))
mlflow.log_param("min cardinality", min(cardinality_dict.values()))
mlflow.log_param("5_columns_with_maximal_cardinalities", top_5)

#### Target Analysis

##### SalePrice distribution

In [ ]:
import matplotlib.pyplot as plt

# Decision Tree does not assume normal distribution of the target
# however, we still remove extreme outliers as they represent mansions
# that are structurally different from the typical housing market

salePriceCol = dataset_df['SalePrice']

fig, ax = plt.subplots()
ax.hist(salePriceCol, bins=50)
plt.show()

print(salePriceCol.skew())

mlflow.log_figure(fig, "saleprice_distribution_before_target_manipulation.png")

##### Outlier removal

In [ ]:
# removing houses whose price exceeds the 99th percentile
# these mansion-level prices are rare and confuse the model

upper = dataset_df['SalePrice'].quantile(0.99)

mansion_prices_df = dataset_df[dataset_df['SalePrice'] > upper]

print(len(mansion_prices_df))

dataset_df = dataset_df[dataset_df['SalePrice'] <= upper]

fig, ax = plt.subplots()
ax.hist(dataset_df['SalePrice'], bins=50)
plt.show()

print(dataset_df['SalePrice'].skew())

mlflow.log_param('outlier_method', 'row_removal')
mlflow.log_param('outlier_upper_quantile', 0.99)
mlflow.log_param('num_rows_removed', len(mansion_prices_df))

#### Cleaning

##### Drop useless columns

In [ ]:
# drop Id
# drop columns below 40% fill threshold
# note: secondary columns are NOT dropped — correlation will decide

outsider_col_names = ['Id']

average_nonNull_entries = (dataset_df.count().sum() / dataset_df.count().size).__round__()
print(f"average number of entries per column: {average_nonNull_entries}")
print(f"column names that have less than 40% of non - null elements out of total average {[i for i in dataset_df.columns if dataset_df[i].count() <= average_nonNull_entries * (40 / 100)]}")
print(f"number of entries accordingly: {[dataset_df[i].count() for i in dataset_df.columns if dataset_df[i].count() <= average_nonNull_entries * (40 / 100)]}")

mlflow.log_param("avg_non_null", average_nonNull_entries)

[outsider_col_names.append(i) for i in dataset_df.columns if dataset_df[i].count() <= average_nonNull_entries * (40 / 100)]

mlflow.log_param("outsider_columns", outsider_col_names)

for col_name in outsider_col_names:
   if dataset_df.columns.str.contains(col_name).any():
       dataset_df = dataset_df.drop(col_name, axis=1, inplace=False)

mlflow.log_param("dropped columns", outsider_col_names)
mlflow.log_param("number of dropped columns", len(outsider_col_names))

dataset_df.count()
print(outsider_col_names)

#### Missing value handling

##### Handling null values in numerical data

In [ ]:
# universal median fill for all numerical columns with nulls

numerical_part_df = dataset_df.select_dtypes(include=['int64', 'float64'])

num_rows = len(dataset_df)

numeric_columns_with_nan = [col for col in numerical_part_df.columns if dataset_df[col].count() != num_rows]

for col in numeric_columns_with_nan:
    dataset_df[col] = dataset_df[col].fillna(dataset_df[col].median())

mlflow.log_param("numeric_null_method", "universal_median")
mlflow.log_param("numeric_columns_with_nan", numeric_columns_with_nan)

##### Handling null values in categorical data

In [ ]:
num_rows = len(dataset_df)

categorical_part_df = dataset_df.select_dtypes(exclude=['int64', 'float64'])

categorical_columns_with_Nan_entries = []

for col in categorical_part_df.columns:
    if dataset_df[col].count() != num_rows:
        categorical_columns_with_Nan_entries.append(col)

print(categorical_columns_with_Nan_entries)

mlflow.log_param("categorical_columns_with_Nan_entries", categorical_columns_with_Nan_entries)

for col in categorical_columns_with_Nan_entries:
    dataset_df[col] = dataset_df[col].fillna("None")

dataset_df.head(5)

mlflow.log_param("categorical_data_null_replacements", ["just string : None"])

#### Feature Engineering

##### pre - FE analysis

In [ ]:
area_cols = [col for col in dataset_df.columns if 'SF' in col or 'Area' in col]
print("Area columns:", area_cols)

time_cols = [col for col in dataset_df.columns if 'Yr' in col or 'Year' in col or 'Mo' in col]
print("Time columns:", time_cols)

bath_cols = [col for col in dataset_df.columns if 'Bath' in col or 'Bsmt' in col]
print("Bath columns:", bath_cols)

##### Creating new features

In [ ]:
dataset_df['TotalSF'] = dataset_df['TotalBsmtSF'] + dataset_df['1stFlrSF'] + dataset_df['2ndFlrSF']
dataset_df['HouseAge'] = dataset_df['YrSold'] - dataset_df['YearBuilt']
dataset_df['RemodAge'] = dataset_df['YrSold'] - dataset_df['YearRemodAdd']
dataset_df['TotalBath'] = dataset_df['FullBath'] + 0.5 * dataset_df['HalfBath'] + dataset_df['BsmtFullBath'] + 0.5 * dataset_df['BsmtHalfBath']
dataset_df['IsRemodeled'] = (dataset_df['YearRemodAdd'] != dataset_df['YearBuilt']).astype(int)

engineered_features = ['TotalSF', 'HouseAge', 'RemodAge', 'TotalBath', 'IsRemodeled']
mlflow.log_param('engineered_features', engineered_features)

#### Turning categorical variables to numeric

##### Pre-encoding analysis

In [ ]:
categorical_part_df = dataset_df.select_dtypes(exclude=['int64', 'float64'])
cardinality_dict = {}
for col in categorical_part_df.columns:
    cardinality_dict[col] = categorical_part_df[col].nunique()
print(cardinality_dict)

##### one hot encoding

In [ ]:
dataset_df = pd.get_dummies(dataset_df, dtype=int, drop_first=True)
dataset_df.info()
mlflow.log_param("number of columns after OHE", dataset_df.columns.size)

#### Feature selection

##### Splitting data to train and test sets

In [ ]:
from sklearn.model_selection import train_test_split

X = dataset_df.drop(columns=['SalePrice'])
y = dataset_df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=42)

mlflow.log_param("train to test proportion", "85% to 15%")

##### Correlation checks

###### Pre-correlation analysis

In [ ]:
columns_with_low_std = [col for col in X_train.columns if X_train[col].std() <= 0.01]
print(columns_with_low_std)

for col in columns_with_low_std:
    X_train.drop(columns=[col], inplace=True)
    X_test.drop(columns=[col], inplace=True)

mlflow.log_param("columns_with_low_std_in_X_train", columns_with_low_std)

###### Checking correlations inbetween columns

In [ ]:
correlation_threshold = 0.85

columns_with_high_correlation = []

abs_corr_between_cols = X_train.corr().abs()

i = 0
j = 0
for Xcol in X_train.columns:
    for Ycol in X_train.columns:
        if i < j and abs_corr_between_cols[Xcol][Ycol] > correlation_threshold:
            if Ycol not in columns_with_high_correlation:
                columns_with_high_correlation.append(Ycol)
        j = j + 1
    j = 0
    i = i + 1

print(columns_with_high_correlation)
print(f"number of columns with absolute correlation more than {correlation_threshold} is {len(columns_with_high_correlation)}")
print(f"total columns in X_train is {len(X_train.columns)}")

mlflow.log_param("correlation_threshold_between_columns", correlation_threshold)
mlflow.log_param("columns_with_high_correlation_between_them", columns_with_high_correlation)

for col in columns_with_high_correlation:
    X_train.drop(columns=[col], inplace=True)
    X_test.drop(columns=[col], inplace=True)

print(len(X_train.columns))

###### Checking correlations with target

In [ ]:
correlation_threshold = 0.1

corr_with_target = X_train.corrwith(y_train).abs().sort_values(ascending=False)

columns_with_low_correlation = [col for col in X_train.columns if corr_with_target[col] <= correlation_threshold]

print(columns_with_low_correlation)
print(f"length of the columns_with_low_correlation is {len(columns_with_low_correlation)}")

mlflow.log_param("correlation_threshold_to_target", correlation_threshold)
mlflow.log_param("number_of_columns_with_low_corr_with_target", len(columns_with_low_correlation))

for col in columns_with_low_correlation:
    X_train.drop(columns=[col], inplace=True)
    X_test.drop(columns=[col], inplace=True)

print(corr_with_target.head(10))

mlflow.log_param("top_10_columns_with_highest_correlation_to_target", corr_with_target.head(10))

len(X_train.columns)

#### Logging X_train structure to reindex test data in inference

In [ ]:
import json
with open('train_columns_decision_tree.json', 'w') as f:
    json.dump(list(X_train.columns), f)

#### Training — Decision Tree

##### Training with Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeRegressor
import numpy as np

max_depth = 7

min_samples_leaf = 10

# optimized model — depth limit plus min_samples_leaf


model = DecisionTreeRegressor(max_depth=max_depth, min_samples_leaf=min_samples_leaf,random_state=42)

mlflow.log_param("model", "DecisionTreeRegressor")
mlflow.log_param("max_depth", max_depth)
mlflow.log_param("min_samples_leaf", min_samples_leaf)

scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')

rmse_scores = -scores
mean_rmse = np.mean(rmse_scores)
std_rmse = np.std(rmse_scores)

print(f"RMSE per fold: {rmse_scores}")
print(f"Mean RMSE: {mean_rmse}")
print(f"Std RMSE: {std_rmse}")

mlflow.log_metric("mean_cv_rmse", mean_rmse)
mlflow.log_metric("std_cv_rmse", std_rmse)

##### Evaluating and logging metrics

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

mlflow.log_metric("test_rmse", test_rmse)
mlflow.log_metric("test_mae", test_mae)
mlflow.log_metric("test_r2", test_r2)

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_test, y_pred, alpha=0.4, s=15)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted')
ax.set_title('Actual vs Predicted')

mlflow.log_figure(fig, "actual_vs_predicted.png")

##### Logging model

In [ ]:
mlflow.sklearn.log_model(model, "decision_tree_optimized")

#### Check underfitting, overfitting

In [ ]:
train_pred = model.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
train_r2 = r2_score(y_train, train_pred)

print(f"Train RMSE: {train_rmse}")
print(f"Test RMSE: {test_rmse}")
print(f"Train R2: {train_r2}")
print(f"Test R2: {test_r2}")

# for a default Decision Tree, train RMSE will be 0 and train R2 will be 1.0
# this is expected — the tree memorizes every training sample
print(f"Difference: {test_rmse - train_rmse}")

mlflow.log_metric("train_rmse", train_rmse)
mlflow.log_metric("train_r2", train_r2)

#### Ending the run

In [ ]:
mlflow.end_run()

In [ ]:
from mlflow.tracking import MlflowClient
from mlflow.tracking import MlflowClient

client = MlflowClient()
run_id = mlflow.last_active_run().info.run_id

client.create_model_version(
    name="house_prices_decision_tree_optimized",
    source=f"runs:/{run_id}/decision_tree_optimized",
    run_id=run_id
)